In [1]:
"""
USE THIS For tod_skill nb to plot more data - 
because saved predict is for TEST dates range and was only for 2026

score_all_dates.py -- one-off: score the FROZEN deployed booster over the full
data span (2022-2026) into a single parquet, for longitudinal / per-day analysis
(window_over_time in tod_skill).

CAVEAT (baked into output)
  The booster was TRAINED on dates < the model's valid_from split. Predictions on
  those dates are IN-SAMPLE: skill there is optimistic on LEVEL. Usable for shape
  / stationarity of a metric over time, NOT as a fair performance number. The
  `in_sample` column marks them; per-year print flags contaminated years.

RULES
  A.1  Identical 5c2 feature path (build_X5b) and frozen artifacts as prod, so p
       on out-of-sample dates matches the deployed pred parquet exactly, and p on
       training dates is what the booster fit.
  A.2  in_sample = date < bundle valid_from (the booster's fit boundary). The iso
       fold (valid_from..train_end) is out-of-sample for the TREES, so those dates
       are fair for the booster though they were used to fit isotonic; marked
       iso_fold separately for full honesty.
  A.3  Output schema matches the deployed pred parquet + {in_sample, iso_fold}:
       bar_index, timestamp, is_target, p, p_cal, in_sample, iso_fold.
"""

import json
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from scipy.signal import lfilter
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, roc_auc_score

from Featurizer import Featurizer
#from stage5b_lgbm_values import (build_X5b, build_meta, _augment_featurizer, ZWARM, BODY_OPEN_COL, BODY_CLOSE_COL, BODY_TAG, BODY_SIGNED, BODY_MAG)

# ---------------------------------------------------------------- CONFIG
FRAME = 6
TAG = "5c2raw_01-2025"
MODEL_PATH = f"data/stage-5/lgbm5b_model_{TAG}_{FRAME}s.joblib"
OUT_PATH = f"data/stage-5/pred_fullspan_{TAG}_{FRAME}s.parquet"

BARS_PATH = f"data/stage-0/bars_{FRAME}s.parquet"
EVENTS_PATH = f"data/stage-0/events_{FRAME}s.parquet"
RAW_PATH = "data/mnq-tick-oscillator-6sec.pqt"
RAW_OHLC_FILE = "data/mnq-ohlc-raw-6sec.pqt"

VALID_FROM = "2025-07-01"          # booster fit boundary for THIS tag (A.2)
TRAIN_END = "2025-12-31"           # iso-fold upper bound



TAUS = (2.0, 6.0, 18.0)

RAW_VALUE_COLS = ["jmaD1", "jmaD2", "tickJmaD1", "tickJmaD2"]                  # signed
RAW_MAG_COLS = ["jmaD1", "jmaD2", "tickJmaD1", "tickJmaD2"]                    # |.|
VALUE_LAGS = (1, 2)                                                            # 5c: extra t-2, t-3

# ----------------------------------------------------------------

In [2]:
#from stage5b_lgbm_values import (build_X5b, build_meta, _augment_featurizer, ZWARM, 
#BODY_OPEN_COL, BODY_CLOSE_COL, BODY_TAG, BODY_SIGNED, BODY_MAG)

BODY_OPEN_COL = "rawOpen"        # set per run (HA or raw column names)
BODY_CLOSE_COL = "rawLast"
BODY_TAG = "raw"               # label only, lands in feature names
BODY_SIGNED = True
BODY_MAG = True

ZWARM = 20                                                                    # S5b.3

def build_lgbm_features(fz, date_from=None, date_to=None):
    blocks = []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        n = S["n"]
        cols = []
        for c in fz.classes:
            P = S["P"][c]
            ind = np.diff(P).astype(np.float64)            # 0/1 event indicator per bar
            occ = np.where(ind > 0, np.arange(n), -1)
            last = np.maximum.accumulate(occ)
            lastm1 = np.concatenate(([-1], last[:-1]))     # last event <= t-1   S5.2
            bsince = np.where(lastm1 >= 0, np.arange(n) - lastm1, np.arange(n) + 1)
            cols.append(bsince[t])
            x = np.concatenate(([0.0], ind[:-1]))          # events shifted to <= t-1
            for tau in TAUS:
                a = np.exp(-1.0 / tau)
                s = lfilter([a], [1.0, -a], x)             # s_t = a*(s_{t-1} + ind_{t-1})
                cols.append(s[t])
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)           # S5.3
        age = np.where(lt >= 0, t - lt, t + 1)
        cols.append(age)
        cols.append(S["tod"][t].astype(np.float64))
        blocks.append(np.stack(cols, 1).astype(np.float32))

    names = []
    for (s, c) in fz.classes:
        names.append(f"{s}|{c}|bsince")
        names += [f"{s}|{c}|ewm{tau:g}" for tau in TAUS]
    names += ["age", "tod"]
    return np.concatenate(blocks), names


In [3]:
def build_lgbm_features(fz, date_from=None, date_to=None):
    blocks = []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        n = S["n"]
        cols = []
        for c in fz.classes:
            P = S["P"][c]
            ind = np.diff(P).astype(np.float64)            # 0/1 event indicator per bar
            occ = np.where(ind > 0, np.arange(n), -1)
            last = np.maximum.accumulate(occ)
            lastm1 = np.concatenate(([-1], last[:-1]))     # last event <= t-1   S5.2
            bsince = np.where(lastm1 >= 0, np.arange(n) - lastm1, np.arange(n) + 1)
            cols.append(bsince[t])
            x = np.concatenate(([0.0], ind[:-1]))          # events shifted to <= t-1
            for tau in TAUS:
                a = np.exp(-1.0 / tau)
                s = lfilter([a], [1.0, -a], x)             # s_t = a*(s_{t-1} + ind_{t-1})
                cols.append(s[t])
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)           # S5.3
        age = np.where(lt >= 0, t - lt, t + 1)
        cols.append(age)
        cols.append(S["tod"][t].astype(np.float64))
        blocks.append(np.stack(cols, 1).astype(np.float32))

    names = []
    for (s, c) in fz.classes:
        names.append(f"{s}|{c}|bsince")
        names += [f"{s}|{c}|ewm{tau:g}" for tau in TAUS]
    names += ["age", "tod"]
    return np.concatenate(blocks), names


def build_meta(fz, date_from=None, date_to=None):
    """Row-aligned with build_lgbm_features (S5.1)."""
    bi, ts, tg, dt = [], [], [], []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        bi.append(S["bar_index"][t])
        ts.append(S["timestamp"][t])
        tg.append(S["tgt"][t])
        dt.append(np.full(len(t), str(S["sess"])))
    return pd.DataFrame({"bar_index": np.concatenate(bi),
                         "timestamp": np.concatenate(ts),
                         "is_target": np.concatenate(tg),
                         "date": np.concatenate(dt)})
    

def _expanding_z(x, warm):
    """Causal expanding z of x[0..n-1] using stats over 0..t-1. z[t]=0 for t<warm."""
    n = len(x)
    csum = np.concatenate(([0.0], np.cumsum(x)))
    csq = np.concatenate(([0.0], np.cumsum(x * x)))
    k = np.arange(n)                                    # count of bars strictly before t
    with np.errstate(invalid="ignore", divide="ignore"):
        mean = np.where(k > 0, csum[:-1] / np.maximum(k, 1), 0.0)
        var = np.where(k > 1, (csq[:-1] - k * mean * mean) / np.maximum(k - 1, 1), 0.0)
        std = np.sqrt(np.maximum(var, 0.0))
        z = np.where((k >= warm) & (std > 0), (x - mean) / std, 0.0)
    return z


def _welford_check(x, warm):
    """Scalar Welford replay of _expanding_z; returns max abs diff (prod parity)."""
    z_vec = _expanding_z(x, warm)
    k = 0
    mean = 0.0
    M2 = 0.0
    z_ref = np.zeros(len(x))
    for t in range(len(x)):
        if k >= warm and M2 > 0:
            z_ref[t] = (x[t] - mean) / np.sqrt(M2 / (k - 1))
        k += 1
        d = x[t] - mean
        mean += d / k
        M2 += d * (x[t] - mean)
    return float(np.max(np.abs(z_vec - z_ref)))


def build_value_features(fz, raw, date_from=None, date_to=None):
    """Value block, row-aligned with build_lgbm_features over the same range."""
    rv = raw.set_index("timestamp")
    blocks = []


    names = ([f"z_{c}_signed" for c in RAW_VALUE_COLS]
                 + [f"z_{c}_mag" for c in RAW_MAG_COLS]
                 + ["z_leg_amp"]
                 + ([f"z_body_{BODY_TAG}_signed"] if BODY_SIGNED else [])
                 + ([f"z_body_{BODY_TAG}_mag"] if BODY_MAG else []))
  
    names = names + [f"{nm}_lag{L}" for L in VALUE_LAGS for nm in names]    

    for S in fz._selected(date_from, date_to):
        n = S["n"]
        ts = pd.DatetimeIndex(S["timestamp"])
        r = rv.reindex(ts)                                                    # S5b.4
        leg_dir = S["leg_dir"]
        leg_start_val = S["leg_start_jma"]

        feats = []
        for c in RAW_VALUE_COLS:                                              # signed
            v = r[c].to_numpy(np.float64) * leg_dir                           # S5b.2
            feats.append(_expanding_z(v, ZWARM))
        for c in RAW_MAG_COLS:                                                # magnitude
            v = np.abs(r[c].to_numpy(np.float64))
            feats.append(_expanding_z(v, ZWARM))

        amp = np.abs(r["JMA"].to_numpy(np.float64) - leg_start_val)
        feats.append(_expanding_z(amp, ZWARM))
        bo = r[BODY_OPEN_COL].to_numpy(np.float64)
        bc = r[BODY_CLOSE_COL].to_numpy(np.float64)
        if BODY_SIGNED:
            feats.append(_expanding_z((bc - bo) * leg_dir, ZWARM))            # confirming-positive
        if BODY_MAG:
            feats.append(_expanding_z(np.abs(bc - bo), ZWARM))

        M = np.stack(feats, 1)                                               # (n, F)
        # shift to t-1 read, then select non-warm rows (must match build_meta order)
        M = np.concatenate([np.zeros((1, M.shape[1])), M[:-1]], 0)            # S5b.2 (t-1)
        # 5c: append lagged copies of the t-1 value block (session-local, zero-padded)
        lagged = [M]
        for L in VALUE_LAGS:
            ML = np.concatenate([np.zeros((L, M.shape[1])), M[:-L]], 0)
            lagged.append(ML)
        M = np.concatenate(lagged, 1)
        t = np.nonzero(~S["warm"])[0]
        Mt = M[t]
        Mt[(t < ZWARM + max(VALUE_LAGS))] = 0.0                               # S5b.5 (+lag warm)
        blocks.append(Mt.astype(np.float32))

    return np.concatenate(blocks), names


def _augment_featurizer(fz, bars):
    """Attach leg_dir and leg_start JMA level per session onto fz.sessions,
    aligned to each session's bar order. Needed by build_value_features."""
    bg = dict(tuple(bars.sort_values("bar_index").groupby("date")))
    for S in fz.sessions:
        g = bg[S["sess"]]
        S["leg_dir"] = g["jma_leg_dir"].to_numpy(np.float64)
        ls = g["bar_index"].to_numpy() - g["bar_index"].to_numpy()[0]
        jma = g["jma"].to_numpy(np.float64)
        # leg_start JMA level: value of jma at the bar that started the current leg
        tgt = g["is_target"].to_numpy()
        starts = np.where(tgt)[0]
        leg_id = np.cumsum(tgt)
        start_idx = np.concatenate(([0], starts))[leg_id]
        S["leg_start_jma"] = jma[start_idx]


def build_X5b(fz, raw, date_from=None, date_to=None):
    Xg, gnames = build_lgbm_features(fz, date_from, date_to)
    Xv, vnames = build_value_features(fz, raw, date_from, date_to)
    assert len(Xg) == len(Xv), (len(Xg), len(Xv))
    return np.hstack([Xg, Xv]), gnames + vnames

In [4]:
def main():
    # body config must match the trained model (A.1) -- assert, don't assume
    assert (BODY_TAG, BODY_SIGNED, BODY_MAG) == ("raw", True, True), "stage5b body config is not the 5c2-raw contract"

    bundle = joblib.load(MODEL_PATH)
    booster = bundle["booster"]
    iso = bundle["iso"]
    iso_x = np.asarray(iso.X_thresholds_, np.float64)
    iso_y = np.asarray(iso.y_thresholds_, np.float64)

    bars = pd.read_parquet(BARS_PATH)
    events = pd.read_parquet(EVENTS_PATH)
    raw = pd.read_parquet(RAW_PATH)
    raw = raw[raw["timestamp"].dt.time >= pd.Timestamp("09:30").time()]
    ohlc = pd.read_parquet(RAW_OHLC_FILE)
    ohlc = ohlc[ohlc["timestamp"].dt.time >= pd.Timestamp("09:30").time()]
    ohlc = ohlc.rename(columns={"Open": "rawOpen", "High": "rawHigh", "Low": "rawLow", "Last": "rawLast"})
    raw = raw.merge(ohlc, on="timestamp", how="left")
    assert raw[["rawOpen", "rawLast"]].notna().all().all()

    fz = Featurizer(bars, events)
    _augment_featurizer(fz, bars)

    # full span: no date filter
    X, names = build_X5b(fz, raw, None, None)                              # A.1
    assert names == list(bundle["feature_names"]), "feature contract mismatch"
    meta = build_meta(fz, None, None)

    p = booster.predict(X.astype(np.float32), num_iteration=booster.best_iteration)
    p_cal = np.interp(p, iso_x, iso_y)

    out = meta[["bar_index", "timestamp", "is_target"]].copy()
    out["p"] = p.astype(np.float32)
    out["p_cal"] = p_cal.astype(np.float32)
    d = out["timestamp"].astype(str).str[:10]
    out["in_sample"] = d < VALID_FROM                                     # A.2
    out["iso_fold"] = (d >= VALID_FROM) & (d <= TRAIN_END)
    out.to_parquet(OUT_PATH, index=False)

    yr = out.assign(year=out["timestamp"].dt.year).groupby("year").agg(
        n=("p", "size"), targets=("is_target", "sum"),
        in_sample=("in_sample", "any"), iso_fold=("iso_fold", "any"))
    print(json.dumps(dict(rows=int(len(out)), out=OUT_PATH,
                          fit_boundary=VALID_FROM), indent=2))
    print(yr.to_string())
    print("\nNOTE: years with in_sample=True are optimistic on level "
          "(booster trained there). Read shape/stationarity only for those.")
    return out


out = main()

{
  "rows": 4450258,
  "out": "data/stage-5/pred_fullspan_5c2raw_01-2025_6s.parquet",
  "fit_boundary": "2025-07-01"
}
           n  targets  in_sample  iso_fold
year                                      
2022  989114   101165       True     False
2023  983192   101597       True     False
2024  989771   102597       True     False
2025  979815   100920       True      True
2026  508366    52198      False     False

NOTE: years with in_sample=True are optimistic on level (booster trained there). Read shape/stationarity only for those.
